# Blade Views Knowledge Inference (Colab Edition)

Run this notebook on Google Colab to query the Blade Knowledge Base.

### 🟢 Prerequisites
1.  **Upload** your `blade_views_chroma_db.zip` file to the files sidebar (root `/content`).
2.  **Run** the cells below in order.

In [ ]:
# 1. Install Dependencies
!pip install -q chromadb FlagEmbedding sentence-transformers groq python-dotenv

In [ ]:
# 2. Unzip Database
import os
import zipfile
import shutil

ZIP_PATH = "/content/blade_views_chroma_db.zip"
EXTRACT_PATH = "/content/blade_views_chroma_db"

if os.path.exists(ZIP_PATH):
    print(f"📦 Found zip at {ZIP_PATH}. Unzipping...")
    # Cleanup previous if exists
    if os.path.exists(EXTRACT_PATH):
        shutil.rmtree(EXTRACT_PATH)
        
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall("/content") # Extracts to /content, creating blade_views_chroma_db folder
    print("✅ Unzipped successfully!")
else:
    if os.path.exists(EXTRACT_PATH):
        print("📂 Database folder already exists. Skipping unzip.")
    else:
        print("❌ ERROR: Zip file not found! Please upload 'blade_views_chroma_db.zip' to the files tab.")

In [ ]:
# 3. Initialize ChromaDB
import chromadb
from chromadb.config import Settings
from FlagEmbedding import BGEM3FlagModel
from dotenv import load_dotenv

load_dotenv()

# Colab Path
CHROMA_DB_PATH = "/content/blade_views_chroma_db"
COLLECTION_NAME = "blade_views_knowledge"

print(f"📂 Checking DB at: {CHROMA_DB_PATH}")

if os.path.exists(CHROMA_DB_PATH):
    # Verify integrity
    files = os.listdir(CHROMA_DB_PATH)
    if "chroma.sqlite3" in files:
        # Connect
        try:
            client = chromadb.PersistentClient(
                path=CHROMA_DB_PATH,
                settings=Settings(anonymized_telemetry=False)
            )
            print(f"✅ Connected to ChromaDB")
            
            # Check collection
            colls = client.list_collections()
            found_names = [c.name for c in colls]
            if COLLECTION_NAME in found_names:
                collection = client.get_collection(name=COLLECTION_NAME)
                print(f"✅ Collection '{COLLECTION_NAME}' loaded with {collection.count()} docs.")
            else:
                print(f"❌ ERROR: Collection '{COLLECTION_NAME}' not found. Available: {found_names}")

        except Exception as e:
            print(f"❌ Connection Error: {e}")
    else:
        print("❌ ERROR: 'chroma.sqlite3' not found inside the folder. Bad zip?")
else:
    print("❌ ERROR: Database directory not found. Did the unzip step work?")

In [ ]:
# 4. Load Model
print("🤖 Loading Embedding Model (BAAI/bge-m3)...")
try:
    model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=False)
    
    from transformers import AutoTokenizer
    model.tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3", use_fast=False)
    
    print("✅ Model loaded successfully")
except Exception as e:
    print(f"❌ Failed to load model: {e}")

## 5. Interactive RAG Query

In [ ]:
# --------------------
# 📝 ENTER QUERY HERE
# --------------------
USER_QUERY = "How does the login form protect against CSRF?"

# --------------------
# Helper Functions Included Here for Durability
# --------------------
from groq import Groq

# 🔑 SET YOUR API KEY HERE
GROQ_API_KEY = "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" 
client_groq = Groq(api_key=GROQ_API_KEY)

def search_blade_views(query, top_k=5):
    query_embedding = model.encode(query, max_length=8192)['dense_vecs'].tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)
    
    formatted_results = []
    if results['documents']:
        for i in range(len(results['documents'][0])):
            formatted_results.append({
                'id': results['ids'][0][i],
                'content': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i] if 'distances' in results else None
            })
    return formatted_results

def generate_rag_answer(query, results):
    context_text = ""
    for i, res in enumerate(results, 1):
        context_text += f"--- Result {i} ---\n"
        context_text += f"File: {res['metadata'].get('file_name')}\n"
        context_text += f"Description: {res['metadata'].get('description')}\n"
        context_text += f"Code Content:\n{res['content'][:1500]}\n\n" # ⬇️ REDUCED CONTEXT LENGTH TO SAVE TOKENS
    
    system_prompt = """You are an expert Laravel Blade developer. 
    Answer the user query based on the provided context. 
    CRITICAL: Provide an EXPLANATION ONLY. Do NOT output code snippets, HTML blocks, or PHP code. 
    Describe the logic and implementation in natural language."""

    user_prompt = f"USER QUERY: {query}\n\nRETRIEVED CONTEXT:\n{context_text}\n\nAnswer:"

    print("⏳ Generating answers (Explanation Only)...")
    
    # ⬇️ SWITCHED TO SMALLER MODEL TO AVOID RATE LIMITS
    # Try 'llama3-8b-8192' or 'mixtral-8x7b-32768' if 70b is limited
    try:
        model_to_use = "llama3-8b-8192" 
        chat_completion = client_groq.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            model=model_to_use,
            temperature=0.0,
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"⚠️ Rate Limit or API Error: {e}"

# --------------------
# Execution Flow
# --------------------
if 'collection' in locals():
    print(f"🔍 Searching for: '{USER_QUERY}'")
    try:
        # ⬇️ REDUCED TOP_K TO 3 TO SAVE TOKENS
        retrieved_docs = search_blade_views(USER_QUERY, top_k=3)
        
        print(f"✅ Found {len(retrieved_docs)} relevant chunks.")
        for i, doc in enumerate(retrieved_docs, 1):
            print(f"   {i}. {doc['metadata'].get('file_name')} ({doc['metadata'].get('section', 'N/A')})")

        print("\n" + "="*50)
        print("🤖 LLM ANSWER (Explanation Only):")
        print("="*50)
        answer = generate_rag_answer(USER_QUERY, retrieved_docs)
        print(answer)
    except NameError as e:
        print(f"❌ NameError: {e}. Please ensure previous cells (Model Loading) ran correctly.")
else:
    print("⚠️ Database not loaded. Run previous cells first.")